In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import os
from PIL import Image
import yaml
import matplotlib.pyplot as plt
import cv2
from collections import OrderedDict
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

print("🔧 Inicializando dispositivo...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo utilizado: {device}")

print("📂 Carregando configuração YAML...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_img_dir = dataset_config["train"]
val_img_dir = dataset_config["val"]
test_img_dir = dataset_config["test"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print(f"✅ Dataset carregado: {len(class_names)} classes")

def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

def replace_extension(file):
    return os.path.splitext(file)[0] + ".txt"

class YoloDataset(Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]

        # Filtrar imagens sem labels
        filtered_imgs = []
        for img in self.imgs:
            label_path = os.path.join(self.label_dir, replace_extension(img))
            if os.path.exists(label_path):
                with open(label_path, 'r') as file:
                    if file.read().strip():
                        filtered_imgs.append(img)
        self.imgs = filtered_imgs

        print(f"📚 Dataset inicializado: {img_dir} com {len(self.imgs)} imagens válidas (com labels)")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, replace_extension(self.imgs[idx]))

        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size

        boxes, labels = load_yolo_labels(label_path, img_width, img_height)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms:
            img = self.transforms(img)

        return img, target

transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])

train_dataset = YoloDataset(train_img_dir, transform)
val_dataset = YoloDataset(val_img_dir, transform)
test_dataset = YoloDataset(test_img_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

class DilatedCNNBackbone(nn.Module):
    def __init__(self):
        super(DilatedCNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, dilation=1),  # normal
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=2, dilation=2),  # dilatada
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=4, dilation=4),  # mais dilatação
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=8, dilation=8),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)  # reduz um pouco a resolução no fim
        self.out_channels = 256

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)  # [B, 256, H/2, W/2]
        return OrderedDict([("0", x)])




print("🔨 Construindo modelo...")
backbone = DilatedCNNBackbone()
anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pooler = MultiScaleRoIAlign(featmap_names=['0'], output_size=7, sampling_ratio=2)
model = torchvision.models.detection.FasterRCNN(backbone, num_classes=nc + 1, rpn_anchor_generator=anchor_generator, box_roi_pool=roi_pooler)
model.to(device)
print("✅ Modelo pronto para treino")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

print("🚀 Iniciando treinamento...")
model.train()

prev_loss = float('inf')
patience = 10
no_improve_epochs = 0
max_epochs = 5

for epoch in range(max_epochs):
    model.train()
    total_loss = 0.0
    total_cls_loss = 0.0
    total_box_loss = 0.0
    total_obj_loss = 0.0
    total_rpn_loss = 0.0

    for batch_idx, (imgs, targets) in enumerate(train_loader, 1):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        total_cls_loss += loss_dict['loss_classifier'].item()
        total_box_loss += loss_dict['loss_box_reg'].item()
        total_obj_loss += loss_dict['loss_objectness'].item()
        total_rpn_loss += loss_dict['loss_rpn_box_reg'].item()

        print(f"📈 Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, "
              f"Loss: {losses.item():.4f} "
              f"(cls: {loss_dict['loss_classifier']:.4f}, "
              f"box: {loss_dict['loss_box_reg']:.4f}, "
              f"obj: {loss_dict['loss_objectness']:.4f}, "
              f"rpn: {loss_dict['loss_rpn_box_reg']:.4f})")

    avg_loss = total_loss / len(train_loader)
    print(f"\n🎯 Epoch {epoch+1} completa. Média de Loss: {avg_loss:.4f} | "
          f"cls: {total_cls_loss:.4f}, box: {total_box_loss:.4f}, "
          f"obj: {total_obj_loss:.4f}, rpn: {total_rpn_loss:.4f}\n")

    if avg_loss + 0.1 < prev_loss:
        print("✅ Melhorando, continuando...")
        prev_loss = avg_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "models/faster_rcnn_yolo_5.pth")
        print("💾 Modelo salvo!")
    else:
        no_improve_epochs += 1
        print("⚠️ Não houve melhoria.")
        if no_improve_epochs >= patience:
            print("🛑 Parando o treinamento por falta de melhoria.")
            break


In [ ]:
import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from sklearn.metrics import precision_score, recall_score, f1_score
import torchvision.transforms as T
from PIL import Image
import yaml
import os
from collections import OrderedDict
from torch.utils.data import DataLoader

# ⚙️ Dispositivo
device = torch.device("cpu")
print(f"🖥️ Usando dispositivo: {device}")

# 📂 Dataset config
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

val_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]

def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

class YoloDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]
        self.transform = transform
        self.imgs = [img for img in self.imgs if os.path.exists(
            os.path.join(self.label_dir, os.path.splitext(img)[0] + ".txt")
        )]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, os.path.splitext(self.imgs[idx])[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = load_yolo_labels(label_path, w, h)
        target = {"boxes": boxes, "labels": labels}

        if self.transform:
            img = self.transform(img)

        return img, target

# 🔁 Transform e DataLoader
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])
val_dataset = YoloDataset(val_dir, transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# 📦 Backbone (DilatedCNN)
class DilatedCNNBackbone(nn.Module):
    def __init__(self):
        super(DilatedCNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, dilation=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=2, dilation=2), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=4, dilation=4), nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=8, dilation=8), nn.ReLU()
        )
        self.pool = nn.MaxPool2d(2)
        self.out_channels = 256

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return OrderedDict([("0", x)])

# 🔧 Montar o modelo
backbone = DilatedCNNBackbone()
anchor_gen = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)

model = FasterRCNN(backbone, num_classes=nc + 1, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
model.load_state_dict(torch.load("models/faster_rcnn_yolo_5.pth", map_location=device))
model.to(device)
model.eval()

# 📊 Avaliação
def evaluate_model(model, dataloader, threshold=0.6):
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, targets in dataloader:
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for t, p in zip(targets, preds):
                gt = t["labels"].cpu().numpy()
                scores = p["scores"].cpu().numpy()
                labels = p["labels"].cpu().numpy()
                pred_filtered = labels[scores > threshold]

                all_classes = set(gt.tolist() + pred_filtered.tolist())
                for cls in all_classes:
                    y_true.append(1 if cls in gt else 0)
                    y_pred.append(1 if cls in pred_filtered else 0)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n📌 Avaliação com limiar > {threshold}")
    print(f"🎯 Precisão: {precision:.3f}")
    print(f"🎯 Revocação: {recall:.3f}")
    print(f"🎯 F1-score: {f1:.3f}")

# 🚀 Executar
evaluate_model(model, val_loader, threshold=0.6)
